### Exploratory data analysis 

In [79]:
import numpy as np
import plotly.express as px
import pandas as pd 
import os
from dash import Dash, dcc, html


In [109]:
df=pd.read_csv("US_honey_dataset (1) (1).csv")
df.shape
df.head()


,Unnamed: 0,state,colonies_number,yield_per_colony,production,stocks,average_price,value_of_production,year
0,0,Alabama,16000,58,928000,28000,62.0,575000,1995
1,1,Arizona,52000,79,4108000,986000,68.0,2793000,1995
2,2,Arkansas,50000,60,3000000,900000,64.0,1920000,1995
3,3,California,420000,93,39060000,4687000,60.0,23436000,1995
4,4,Colorado,45000,60,2700000,1404000,68.0,1836000,1995


In [81]:
# Check null values 
df.isnull().sum()


Unnamed: 0             0
state                  0
colonies_number        0
yield_per_colony       0
production             0
stocks                 0
average_price          0
value_of_production    0
year                   0
dtype: int64

In [82]:
# Check duplicates values 
print(df.duplicated().sum())

# Check the data types 
print(df.info())


0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           1115 non-null   int64  
 1   state                1115 non-null   object 
 2   colonies_number      1115 non-null   int64  
 3   yield_per_colony     1115 non-null   int64  
 4   production           1115 non-null   int64  
 5   stocks               1115 non-null   int64  
 6   average_price        1115 non-null   float64
 7   value_of_production  1115 non-null   int64  
 8   year                 1115 non-null   int64  
dtypes: float64(1), int64(7), object(1)
memory usage: 78.5+ KB
None


In [83]:
# Statistical summary of the dataset
print(df.describe())

       Unnamed: 0  colonies_number  yield_per_colony    production  \
count  1115.00000      1115.000000       1115.000000  1.115000e+03   
mean    557.00000     62438.565022         59.743498  2.851268e+06   
std     322.01708     92648.175955         19.940500  5.561202e+06   
min       0.00000      2000.000000         19.000000  1.200000e+04   
25%     278.50000      9000.000000         45.000000  2.460000e+05   
50%     557.00000     26000.000000         57.000000  8.280000e+05   
75%     835.50000     69000.000000         71.000000  2.700000e+06   
max    1114.00000    550000.000000        155.000000  3.906000e+07   

             stocks  average_price  value_of_production         year  
count  1.115000e+03    1115.000000         1.115000e+03  1115.000000  
mean   1.172625e+06     140.623076         5.667412e+06  2007.740807  
std    2.049556e+06     107.011544         9.459460e+06     7.823002  
min    9.000000e+03       1.300000         1.060000e+05  1995.000000  
25%    1.12500

In [84]:
df['state'].value_counts()

state
Alabama          27
Arizona          27
Arkansas         27
California       27
Colorado         27
Florida          27
Georgia          27
Hawaii           27
Idaho            27
Illinois         27
Indiana          27
Iowa             27
Kansas           27
Louisiana        27
Maine            27
Michigan         27
Mississippi      27
Minnesota        27
Montana          27
Missouri         27
NorthCarolina    27
NewYork          27
Nebraska         27
NewJersey        27
Pennsylvania     27
Oregon           27
Ohio             27
NorthDakota      27
Washington       27
WestVirginia     27
Vermont          27
Virginia         27
Tennessee        27
SouthDakota      27
Utah             27
Texas            27
Wisconsin        27
Wyoming          27
Kentucky         26
NewMexico        18
Nevada           15
SouthCarolina    12
Maryland          9
Oklahoma          9
Name: count, dtype: int64

In [85]:
# Advanced EDA Visualizations using Plotly

# 1. HONEY PRODUCTION TREND OVER TIME BY TOP STATES
top_states = df.groupby('state')['production'].sum().nlargest(8).index
df_top_states = df[df['state'].isin(top_states)]

fig1 = px.line(
    df_top_states,
    x='year',
    y='production',
    color='state',
    title='<b>Honey Production Trend Over Years - Top 8 States</b>',
    labels={'production': 'Production (lbs)', 'year': 'Year', 'state': 'State'},
    markers=True,
    hover_data={'production': ':,.0f'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig1.update_layout(
    hovermode='x unified',
    height=500,
    template='plotly_dark',
    font=dict(size=11),
    title_font_size=14
)
fig1.show()


#### 1. Honey Production Trend Over Years - Top 8 States
- Shows production trajectories for the eight states with the highest total production.
- Highlights South Dakota lead production over time and the output is falling in long term for all states.
- Useful for comparing long-term state performance and spotting trend shifts.
- Earlier California followed by North Dakota were leading Production but their production declined over time

In [102]:
from dash import Dash, dcc, html, Input, Output
app = Dash(__name__)
app.layout = html.Div([
    html.H2('Honey Stocks and Production Dashboard'),
    dcc.Dropdown(
        id='year-dropdown',
        options=[{'label': str(year), 'value': year} for year in sorted(df['year'].unique())],
        value=df['year'].max(),
        clearable=False,
        style={'width': '320px', 'margin-bottom': '20px'}
    ),
    html.Div([
        dcc.Graph(id='bubble-chart', style={'width': '100%', 'display': 'inline-block'})
    ])
])

@app.callback(
    Output('bubble-chart', 'figure'),
    Input('year-dropdown', 'value')
)
def update_charts(selected_year):
    df_year = df[df['year'] == selected_year]

    bubble_fig = px.scatter(
        df_year,
        x='colonies_number',
        y='yield_per_colony',
        size='production',
        color='average_price',
        hover_name='state',
        title=f'<b>Colonies vs Yield with Production Scale (& Price) - {selected_year}</b>',
        labels={
            'colonies_number': 'Number of Colonies',
            'yield_per_colony': 'Yield per Colony (lbs)',
            'production': 'Total Production (lbs)',
            'average_price': 'Average Price ($/lb)'
        },
        color_continuous_scale='Viridis',
        size_max=60
    )
    bubble_fig.update_layout(height=550, template='plotly_white', font=dict(size=11), title_font_size=14)


    return bubble_fig
app.run(mode='inline',port=8054)
       

#### 2. Colonies vs Yield with Production Scale (& Price) - Year Selector
- Interactive bubble chart filtered by selected year.
- X-axis: number of colonies; Y-axis: yield per colony; bubble size: production volume; bubble color: average price.
- Reveals state-level efficiency and how price relates to production scale in a single year.
- In 2021, North Dakota has highest no. of colonies but Yield per colony is moderate, depicting less efficiency and low average price, while states such as Hawai (93), Mississipi, Ohio are more efficient as they have high yield per colony.



In [87]:
# 3. AVERAGE PRICE EVOLUTION - BOX PLOT BY DECADE
df['decade'] = (df['year'] // 10 * 10).astype(str)

fig3 = px.box(
    df,
    x='decade',
    y='average_price',
    title='<b>Honey Price Distribution Across Decades</b>',
    labels={'average_price': 'Average Price ($/lb)', 'decade': 'Decade'},
    color='decade',
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig3.update_layout(height=500, template='plotly_dark', font=dict(size=11), title_font_size=14)
fig3.show()


#### 3. Honey Price Distribution Across Decades
- Box plot of average honey price by decade.
- Shows price range, median, and outliers for each decade.
- Useful for identifying long-term price inflation and decade-specific volatility.
- While the prices were higher in 2010, they declined in 2020 decade

In [88]:

# 4. PRODUCTION VALUE vs ACTUAL PRODUCTION - SCATTER WITH TRENDLINE
fig4 = px.scatter(
    df,
    x='production',
    y='value_of_production',
    color='year',
    trendline='ols',
    hover_name='state',
    title='<b>Total Production Value vs Physical Production Volume</b>',
    labels={
        'production': 'Production Volume (lbs)',
        'value_of_production': 'Production Value ($)',
        'year': 'Year'
    },
    color_continuous_scale='Blues',
    opacity=0.6
)
fig4.update_layout(height=600, template='plotly_dark', font=dict(size=11), title_font_size=14)
fig4.show()


#### 4. Total Production Value vs Physical Production Volume
- Scatter plot with a trendline between production volume and value of production.
- Demonstrates whether higher production generally translates to higher production value.
- Helps detect pricing or value anomalies in production-heavy states.
- It shows Higher production volume doesn't necessarily mean higher production value. Slope of OLS is flatter. In recent years, States have low production volume but have higher value. 
- OLS is not suitable here because of outliers.

In [89]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

app = Dash(__name__)

years = sorted(df["year"].unique())

app.layout = html.Div(
    [
        html.H1(
            "US Honey Stocks Analysis",
            style={"textAlign": "center"}
        ),

        html.Label("Select Year:", style={"fontWeight": "bold"}),

        dcc.Dropdown(
            id="year-dropdown",
            options=[{"label": str(year), "value": year} for year in years],
            value=max(years),
            clearable=False,
            style={"width": "300px"}
        ),

        dcc.Graph(id="stocks-pie-chart")
    ],
    style={"padding": "20px"}
)


@app.callback(
    Output("stocks-pie-chart", "figure"),
    Input("year-dropdown", "value")
)
def update_pie_chart(selected_year):

    df_top_stocks = (
        df[df["year"] == selected_year]
        .nlargest(10, "stocks")
    )

    fig = px.pie(
        df_top_stocks,
        values="stocks",
        names="state",
        title=f"<b>Honey Stocks Distribution - Top 10 States ({selected_year})</b>",
        color_discrete_sequence=px.colors.qualitative.Antique
    )

    fig.update_traces(
        textposition="inside",
        textinfo="percent+label",
        hovertemplate="<b>%{label}</b><br>Stocks: %{value:,.0f}<extra></extra>"
    )

    fig.update_layout(
        height=650,
        template="plotly_dark",
        font=dict(size=12),
        title_font_size=18,
        legend_title="State"
    )

    return fig


if __name__ == "__main__":
    app.run(debug=True,port=8052)

#### 5. Honey Stocks Distribution - Top 10 States
- Interactive pie chart showing stocks share for the top 10 states in the chosen year.
- Makes it easy to see which states hold the largest honey inventories.
- The year selector lets you compare stock distribution across different years.
- SouthDakota, NorthDakota, Michigan, Montana have highest stock distribution together comprising nearly 60 percent. Michigan recently overtook Montana, California, Minnesota in 2021. Share of NorthDakota reduced in 2021.

In [112]:
# 5. HONEY PRODUCTION HIERARCHY - SUNBURST CHART
df['decade'] = (df['year'] // 10 * 10).astype(str)
fig = px.sunburst(
    df,
    path=["decade","state"],
    values="production",
    color="average_price",
    title="Honey Production Hierarchy"
)
fig.update_layout(height=600, template="plotly_dark", font=dict(size=11), title_font_size=14)
fig.show()

#### 6. Honey Production Sunburn - Top States Over Time
- Visualizes temporal production patterns and relative intensity with color.
- Good for spotting which states consistently dominate and when production peaks occur.
- California, NorthDakota were Highest producers in 1990 and 2000 while the SouthDakota is top producer in 2020 but the average price of honey is low now.


In [91]:
# 7. YIELD EFFICIENCY ANALYSIS - BAR CHART
avg_yield_by_state = df.groupby('state').agg({
    'yield_per_colony': 'mean',
    'production': 'sum'
}).reset_index().sort_values('yield_per_colony', ascending=False).head(15)

fig7 = px.bar(
    avg_yield_by_state,
    x='state',
    y='yield_per_colony',
    color='yield_per_colony',
    title='<b>Average Honey Yield Efficiency - Top 15 States</b>',
    labels={
        'yield_per_colony': 'Avg Yield per Colony (lbs)',
        'state': 'State'
    },
    color_continuous_scale='Greens',
    text='yield_per_colony'
)
fig7.update_traces(textposition='outside')
fig7.update_layout(height=500, template='plotly_dark', font=dict(size=11), title_font_size=14)
fig7.show()



#### 7. Average Honey Yield Efficiency - Top 15 States
- Bar chart of average yield per colony for the top 15 states.
- Highlights which states achieve the highest productivity per colony.
- Useful for performance benchmarking beyond raw production volume.
- Hawaii, Louisiana, Mississipi are highly efficient.

In [92]:
# 8. PRICE CORRELATION WITH PRODUCTION - BUBBLE MAP BY STATE
avg_metrics = df.groupby('state').agg({
    'production': 'mean',
    'average_price': 'mean',
    'value_of_production': 'mean'
}).reset_index().sort_values('average_price', ascending=False).head(12)

fig8 = px.scatter(
    avg_metrics,
    x='average_price',
    y='value_of_production',
    size='production',
    hover_name='state',
    title='<b>Price vs Production Value with Volume Scale</b>',
    labels={
        'average_price': 'Average Price ($/lb)',
        'value_of_production': 'Avg Production Value ($)',
        'production': 'Avg Production (lbs)'
    },
    color_discrete_sequence=['#FF6B6B'],
    size_max=50
)
fig8.update_layout(height=600, template='plotly_dark', font=dict(size=11), title_font_size=14)
fig8.show()

#### 8. Price vs Production Value with Volume Scale
- Bubble chart comparing average price to average production value.
- Bubble size shows production volume, so large and expensive producers stand out.
- Ohio and South Carolina have highest production value but low pricing power. Similarly, Virginia have highest pricing power.

In [114]:
us_state_abbrev = {
    "Alabama":"AL",
    "Arizona":"AZ",
    "Arkansas":"AR",
    "California":"CA",
    "Colorado":"CO",
    "Florida":"FL",
    "Georgia":"GA",
    "Hawaii":"HI",
    "Idaho":"ID",
    "Illinois":"IL",
    "Indiana":"IN",
    "Iowa":"IA",
    "Kansas":"KS",
    "Kentucky":"KY",
    "Louisiana":"LA",
    "Maine":"ME",
    "Maryland":"MD",
    "Michigan":"MI",
    "Minnesota":"MN",
    "Mississippi":"MS",
    "Missouri":"MO",
    "Montana":"MT",
    "Nebraska":"NE",
    "Nevada":"NV",
    "New Jersey":"NJ",
    "New Mexico":"NM",
    "New York":"NY",
    "North Carolina":"NC",
    "North Dakota":"ND",
    "Ohio":"OH",
    "Oklahoma":"OK",
    "Oregon":"OR",
    "Pennsylvania":"PA",
    "South Carolina":"SC",
    "South Dakota":"SD",
    "Tennessee":"TN",
    "Texas":"TX",
    "Utah":"UT",
    "Vermont":"VT",
    "Virginia":"VA",
    "Washington":"WA",
    "West Virginia":"WV",
    "Wisconsin":"WI",
    "Wyoming":"WY"
}

df["state_code"] = df["state"].map(us_state_abbrev)

In [115]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

app = Dash(__name__)

years = sorted(df["year"].unique())

app.layout = html.Div(
    [
        html.H1(
            "US Honey Stocks Analysis",
            style={"textAlign": "center"}
        ),

        html.Label("Select Year:", style={"fontWeight": "bold"}),

        dcc.Dropdown(
            id="year-dropdown",
            options=[{"label": str(year), "value": year} for year in years],
            value=max(years),
            clearable=False,
            style={"width": "300px"}
        ),

        dcc.Graph(id="choropleth")
    ],
    style={"padding": "20px"}
)


@app.callback(
    Output("choropleth", "figure"),
    Input("year-dropdown", "value")
)
def update_choropleth(selected_year):

    fig = px.choropleth(
    df[df["year"] == selected_year],
    locations="state_code",
    locationmode="USA-states",
    color="production",
    hover_name="state",
    scope="usa",
    color_continuous_scale="YlOrRd",
    title=f"Honey Production by State ({selected_year})"
)
    return fig


if __name__ == "__main__":
    app.run(debug=True,port=8050)


#### 10. Honey Production by State Choropleth
- Map of state-level production for a selected year.
- Colors indicate production intensity across US states.
- Useful for geographic comparison and identifying regional production hubs.


In [95]:

import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

app = Dash(__name__)

years = sorted(df["year"].unique())

app.layout = html.Div(
    [
        html.H1(
            "US Honey Stocks Analysis",
            style={"textAlign": "center"}
        ),

        html.Label("Select Year:", style={"fontWeight": "bold"}),

        dcc.Dropdown(
            id="year-dropdown",
            options=[{"label": str(year), "value": year} for year in years],
            value=max(years),
            clearable=False,
            style={"width": "300px"}
        ),

        dcc.Graph(id="scatter")
    ],
    style={"padding": "20px"}
)


@app.callback(
    Output("scatter", "figure"),
    Input("year-dropdown", "value")
)
def scatter(selected_year):
    fig = px.scatter(
    df[df["year"] == selected_year],
    x="production",
    y="stocks",
    size="colonies_number",
    color="average_price",
    hover_name="state",
    title=f"Production vs Stocks ({selected_year})"
)

    return fig 

if __name__ == "__main__":
    app.run(debug=True,port=8055)



#### Production vs Stocks
- It shows that stocks and Production have high positive correlation
- States which have higher production tend to hold more stock

In [96]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

app = Dash(__name__)

years = sorted(df["year"].unique())

app.layout = html.Div(
    [
        html.H1(
            "US Honey Stocks Analysis",
            style={"textAlign": "center"}
        ),

        html.Label("Select Year:", style={"fontWeight": "bold"}),

        dcc.Dropdown(
            id="year-dropdown",
            options=[{"label": str(year), "value": year} for year in years],
            value=max(years),
            clearable=False,
            style={"width": "300px"}
        ),

        dcc.Graph(id="treemap")
    ],
    style={"padding": "20px"}
)


@app.callback(
    Output("treemap", "figure"),
    Input("year-dropdown", "value")
)
def treemap(selected_year):
    fig = px.treemap(
    df[df["year"] == selected_year],
    path=["state"],
    values="production",
    color="average_price",
    title="Production Treemap"
)

    return fig 

if __name__ == "__main__":
    app.run(debug=True,port=8051)



#### Production heatmap
- It shows the proportion of production in states over years while color shows average price variation

In [97]:
fig = px.line(
    df.groupby("year")["average_price"].mean().reset_index(),
    x="year",
    y="average_price",
    markers=True,
    title="Average Honey Price Over Time"
)
fig.show()

#### Avg Honey price over time
- Honey price was upward sloping until 2017
- It fell sharply after 2017